# Generación de Datos Sintéticos y Seeder de Base de Datos para FinanceAI
Este notebook emula el comportamiento de gasto de usuarios a lo largo de **365 días hasta el 21 de agosto de 2026** (del **22 de agosto de 2025 al 21 de agosto de 2026**), aplicando la lógica econométrica de `1_simulation.ipynb` con **restricción presupuestaria mensual cerrada** (sin crédito ni arrastre de dinero).

### Configuración del Seeder
*   **Usuarios**: 10 usuarios con distribución salarial triangular, emails normalizados y contraseñas cifradas en **BCrypt** (`password1234.`).
*   **Hash BCrypt de contraseña**: `$2a$10$oZqN23owgFlc8mn.mHavludNTpKhoa7nBhIcgp2iBlVKBG5IF.Abe`
*   **Transacciones**: 600 transacciones por cada usuario a lo largo de los 365 días (total: **6,000 transacciones**).
*   **Ventana temporal viva**: Del 22 de agosto de 2025 al 21 de agosto de 2026 para permitir visualización en tiempo real en dashboards.
*   **Restricción Presupuestaria**: El gasto de cada período se acota estrictamente a su `ingreso_mensual` (sin exceder jamás el salario y con margen de ahorro según el perfil).
*   **Comportamiento**: 10 categorías oficiales, elasticidad por poder adquisitivo, distribución triangular de montos por categoría y perfiles diferenciados (austeros, estándar y derrochadores).


In [ ]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta
import unicodedata
import os

# Fijar semillas para reproducibilidad
SEED = 42
np.random.seed(SEED)
random.seed(SEED)

output_dir = './data'
os.makedirs(output_dir, exist_ok=True)
print(f"Directorio de salida configurado en: {os.path.abspath(output_dir)}")


## 1. Generación de Usuarios (10 Usuarios)
*   **Ingresos**: Distribución triangular con mínimo de 1.500, moda en 1.800 y máximo de 5.000 USD.
*   **Seguridad**: Contraseña fija pre-hasheada con BCrypt (`password1234.`) usando el hash verificado `$2a$10$oZqN23owgFlc8mn.mHavludNTpKhoa7nBhIcgp2iBlVKBG5IF.Abe`.
*   **Fechas de creación**: Registrados en 2024 / principios de 2025 para asegurar que todas las transacciones sean cronológicamente válidas y posteriores a la creación de la cuenta.


In [ ]:
num_usuarios = 10
transacciones_por_usuario = 600

nombres = [
    'Ana Garcia', 'Juan Perez', 'Sofia Rodriguez', 'Pedro Gomez', 'Laura Martinez',
    'Diego Lopez', 'Valentina Diaz', 'Carlos Fernandez', 'Camila Alvarez', 'Luis Romero'
]

def generar_email(nombre_completo):
    nombre_limpio = ''.join(c for c in unicodedata.normalize('NFD', nombre_completo)
                  if unicodedata.category(c) != 'Mn')
    nombre_limpio = nombre_limpio.lower().replace(' ', '')
    return f"{nombre_limpio}@prueba.com"

# Distribución triangular de sueldos idéntica a 1_simulation
ingresos = np.round(np.random.triangular(1500, 1800, 5000, size=num_usuarios), 2)

# Fechas de creación previas a la ventana de transacciones (2024 / principios 2025)
fechas_creacion = [
    (datetime(2024, 6, 1) + timedelta(
        days=int(random.randint(0, 200)),
        hours=int(random.randint(0, 23)),
        minutes=int(random.randint(0, 59))
    )).strftime("%Y-%m-%d %H:%M:%S")
    for _ in range(num_usuarios)
]

# Hash BCrypt verificado de 'password1234.'
hash_bcrypt = "$2a$10$oZqN23owgFlc8mn.mHavludNTpKhoa7nBhIcgp2iBlVKBG5IF.Abe"

usuarios = []
for i in range(num_usuarios):
    usuarios.append({
        "id": i + 1,
        "nombre": nombres[i],
        "email": generar_email(nombres[i]),
        "password": hash_bcrypt,
        "ingreso_mensual": ingresos[i],
        "fecha_creacion": fechas_creacion[i],
        "activo": True
    })

df_usuarios = pd.DataFrame(usuarios)
print(f"Se han generado {len(df_usuarios)} usuarios.")
display(df_usuarios[['id', 'nombre', 'email', 'ingreso_mensual', 'fecha_creacion']])


## 2. Generación de Transacciones (365 días: 2025-08-22 al 2026-08-21)
Reglas aplicadas:
1. **Sin crédito ni arrastre**: En cada período mensual, el gasto total del usuario está acotado por su salario mensual:
   - **Austeros**: Gastan entre 45% y 65% de su sueldo (alto ahorro/inversión).
   - **Estándar**: Gastan entre 70% y 85% de su sueldo.
   - **Derrochadores**: Gastan entre 92% y 98% de su sueldo (viven al límite sin excederlo).
2. **Distribución mensual**: 600 transacciones repartidas proporcionalmente en los meses completos y parciales del año móvil.
3. **Ponderación y elasticidad de montos** acorde a los rangos calibrados en `1_simulation.ipynb`.


In [ ]:
diccionario_conceptos = {
    'Alimentacion': [
        'supermercado coto', 'verduleria el sol', 'carniceria central', 'almacen san martin', 
        'compra panaderia', 'supermercado carrefour', 'compras fiambreria', 'compra dia',
        'super', 'super chino', 'kiosco', 'minimercado', 'dietetica', 'mercado de barrio',
        'jumbo', 'disco', 'changomas', 'compra hipermercado', 'fruteria', 'papas fritas',
        'rotiseria', 'rotiseria de barrio', 'polleria', 'pescaderia',
        'pizzeria', 'cotillon', 'distribuidora alimentos', 'feria franciscana',
        'autoservicio', 'verduleria la huerta', 'compra coca cola',
        'supermercado lider', 'carniceria de barrio', 'distribuidora bebidas', 'fiambreria y quesos',
        'compra de verdura', 'almacen de campo', 'compra de lacteos', 'polleria del centro',
        'panaderia de barrio', 'gastos super', 'compras minimercado', 'panaderia artesanal',
        'verduleria express', 'pescaderia central', 'rotiseria express', 'almacen express'
    ],
    'Educacion': [
        'cuota universidad', 'compra libros', 'curso de programacion', 'matricula colegio', 
        'utiles escolares', 'taller ingles', 'cuota jardin de infantes', 'pago facultad', 'facu',
        'fotocopias facultad', 'suscripcion udemy', 'coderhouse', 'libreria escolar',
        'cuota instituto', 'clases particulares', 'taller pintura', 'curso de idiomas',
        'examen certificado', 'derecho a examen', 'cuota maternal', 'taller ceramica',
        'capacitacion python', 'cuota posgrado', 'bootcamp', 'curso domestika',
        'libreria tecnica', 'simposio', 'curso de data science',
        'taller de robotica', 'derecho examen', 'fotocopias apuntes', 'libreria de barrio',
        'cuota colegio privado', 'clases de apoyo', 'taller de teatro', 'capacitacion online',
        'curso de marketing', 'cuota jardin privado', 'arancel universidad', 'inscripcion curso',
        'compra cuaderno', 'libreria universitaria', 'taller de musica', 'clases de guitarra'
    ],
    'Electrodomesticos': [
        'compra heladera', 'lavarropas fravega', 'televisor garbarino', 'microondas musimundo', 
        'licuadora philips', 'pava electrica', 'aire acondicionado', 'compra de pc',
        'celular nuevo', 'tablet', 'auriculares', 'notebook', 'ventilador', 'estufa',
        'cafetera', 'plancha', 'cetrogar', 'megatone', 'electrodomesticos varios',
        'freidora de aire', 'aspiradora robot', 'horno electrico', 'tostadora',
        'extractor de aire', 'lavavajillas', 'minipimer', 'sandwichera',
        'parlante bluetooth',
        'parlante portatil', 'televisor smart', 'notebook gamer', 'camara fotografica',
        'placa de video', 'plancha de pelo', 'secador de pelo', 'monitor gamer',
        'impresora multifuncion', 'consola de juegos', 'molinillo de cafe', 'horno con anfe',
        'pava de acero', 'heladera inverter', 'lavarropas inverter', 'licuadora de mano'
    ],
    'Inversion': [
        'compra dolares', 'fondo comun inversion', 'plazo fijo', 'cedears', 'bonos del estado', 
        'acciones ypf', 'transferencia broker', 'balanz', 'bull market', 'compra mep',
        'dolar ahorro', 'criptomonedas', 'binance', 'lemon cash', 'fci', 'bonos al30',
        'inversion portfolio', 'ahorro mensual', 'compra usdt', 'plazo fijo crypto', 
        'bybit', 'iol', 'inversiu', 'cauciones', 
        'obligaciones negociables', 'compra eth', 'compra btc', 'cocos capital',
        'balanz', 'bullmarket', 'cuenta comitente',
        'compra btc crypto', 'acciones galicia', 'bonos soberanos', 'fondo comun fci',
        'transferencia lemon', 'deposito reba', 'dolar bolsa mep', 'inversion en cedear',
        'operacion de cambio', 'compra acciones us', 'transferencia mercadopago', 'inversion en bonos',
        'plazo fijo uva', 'fondos de inversion', 'cripto usdt', 'transferencia a broker'
    ],
    'Ocio': [
        'salida cine', 'suscripcion netflix', 'cena restaurante', 'entradas recital', 'alquiler auto', 
        'cerveceria', 'suscripcion spotify', 'juegos steam', 'cafeteria', 'bar',
        'mcdonalds', 'burger king', 'salida teatro', 'suscripcion prime video',
        'juegos playstation', 'heladeria', 'salida boliche', 'entradas futbol', 
        'escape room', 'bowling', 'alquiler cancha', 'parque de atracciones',
        'stand up', 'merienda cafe', 'teatro', 'cuota gimnasio',
        'suscripcion disney', 'fiesta privada', 'compra fuegos artificiales',
        'salida a cenar', 'ticket de cine', 'entradas para teatro', 'merienda en cafe',
        'pinta de cerveza', 'cuota de gimnasio', 'suscripcion spotify', 'suscripcion hbo',
        'juego de mesa', 'salida a bailar', 'paseo en barco', 'alquiler de bicicleta',
        'entrada a museo', 'ticket para recital', 'alquiler cancha futbol', 'pago de delivery'
    ],
    'Salud': [
        'estudios clinicos', 'compra farmacia', 'consulta medica', 'cuota prepaga', 
        'medicamentos', 'dentista', 'analisis sangre', 'optica', 'osde', 'swiss medical',
        'farmacity', 'farmacia', 'laboratorio', 'psicologo', 'kinesiologia', 'lentes',
        'galeno', 'urgencia medica', 'vacunas',
        'odontologo', 'remedios', 'psiquiatra', 'traumatologo',
        'pediatra', 'estudios vista', 'ecografia', 'radiografia',
        'paracetamol', 'medicacion cronica',
        'consulta odontologica', 'analisis de laboratorio', 'medicamentos recetados', 'sesion de terapia',
        'atencion medica', 'farmacia de turno', 'compra de remedios', 'sesion kinesiologia',
        'consulta dermatologo', 'plano de plantilla', 'estudio de radiografia', 'ecografia abdominal',
        'cuota de prepaga', 'compra de medicamentos', 'receta medica', 'estudio ecocardiograma'
    ],
    'Servicios': [
        'factura luz edesur', 'abono internet', 'servicio agua', 'factura gas', 
        'telefonia movil', 'impuesto municipal', 'abl', 'rentas', 'personal', 'movistar',
        'claro', 'telecentro', 'fibertel', 'metrogas', 'edenor', 'aysa', 'luz', 'gas',
        'seguro hogar', 'cablevision', 'directv',
        'flow', 'iplan', 'movistar fibra', 'ipuc',
        'ecogas', 'litoral gas', 'edet', 'epe',
        'seguro vida', 'patente auto',
        'factura de luz', 'factura de agua', 'abono de celular', 'servicio de internet',
        'impuesto inmobiliario', 'patente municipal', 'suscripcion de cable', 'servicio de gas',
        'expensas de departamento', 'seguro contra incendio', 'mantenimiento de red', 'servicio de alarma',
        'impuesto de sellos', 'tasa municipal', 'servicio de cloacas', 'abono telefonia'
    ],
    'Transporte': [
        'viaje uber', 'colectivo', 'carga sube', 'combustible ypf', 'peaje autopista', 
        'taxi', 'viaje cabify', 'combustible shell', 'sube', 'axion', 'puma energy',
        'estacionamiento', 'lavadero auto', 'seguro auto', 'tren', 'subte', 'boleto',
        'pasaje micro', 'didi', 'vtv',
        'mantenimiento auto', 'cambio de aceite', 'alineacion y balanceo', 'pasaje avion',
        'peaje caba', 'remis', 'service oficial', 'cubiertas',
        'mecano', 'parche rueda',
        'carga tarjeta sube', 'peaje de autopista', 'combustible nafta', 'viaje en taxi',
        'pasaje de colectivo', 'boleto de tren', 'mantenimiento de auto', 'estacionamiento medido',
        'service de auto', 'peaje acceso norte', 'pasaje de micro', 'viaje en remis',
        'lavadero de autos', 'cambio de cubiertas', 'parche de cubierta', 'cambio de filtro'
    ],
    'Vestimenta': [
        'compra zapatillas', 'pantalon jean', 'remera algodon', 'campera invierno', 
        'ropa deportiva', 'local indumentaria', 'zapatos', 'zara', 'dexter', 'moov',
        'compra ropa', 'ropa interior', 'buzo', 'camisa', 'shopping', 'indumentaria',
        'accesorios moda', 'zapatillas nike', 'adidas', 'ropa invierno',
        'medias', 'malla', 'pollera', 'vestido',
        'barbijo', 'lencería', 'pijama', 'camisa blanca',
        'calzado seguridad', 'ojotas',
        'remera de algodon', 'pantalon de vestir', 'zapatillas deportivas', 'campera de cuero',
        'ropa para entrenar', 'buzo con capucha', 'accesorios de moda', 'medias deportivas',
        'camisa manga corta', 'vestido de fiesta', 'saquito de lana', 'calzado deportivo',
        'traje de baño', 'pijama de invierno', 'ojotas de playa', 'ropa de trabajo'
    ],
    'Vivienda': [
        'pago alquiler', 'expensas edificio', 'servicio plomeria', 'ferreteria', 
        'pintura habitacion', 'reparacion electrica', 'materiales construccion', 'alquiler',
        'expensas', 'easy', 'sodimac', 'cerrajero', 'gasista', 'muebles',
        'decoracion', 'inmobiliaria', 'corredor inmobiliario', 'limpieza',
        'pintura casa', 'plomero', 'reparacion calefon', 'electricista',
        'fumigacion', 'arreglos persiana', 'cuota hipoteca', 'deposito garantia',
        'expensas extraordinarias', 'flete mudanza',
        'pago de alquiler', 'expensas ordinarias', 'servicio de plomeria', 'pintura para pared',
        'compra de muebles', 'flete por mudanza', 'honorarios inmobiliaria', 'deposito de alquiler',
        'reparacion de gas', 'trabajo de carpinteria', 'limpieza de departamento', 'servicio de cerrajeria',
        'arreglo de persiana', 'cuota de hipoteca', 'articulos de limpieza', 'mantenimiento de casa'
    ]
}

# Rangos base de montos idénticos a 1_simulation
rangos_montos = {
    'Alimentacion': (15, 100), 'Educacion': (30, 200), 'Electrodomesticos': (200, 400), 'Inversion': (100, 300),
    'Ocio': (20, 80), 'Salud': (40, 200), 'Servicios': (10, 80), 'Transporte': (3, 200),
    'Vestimenta': (10, 300), 'Vivienda': (100, 450)
}

categorias = list(diccionario_conceptos.keys())
prob_categorias = [0.15, 0.05, 0.05, 0.10, 0.10, 0.05, 0.20, 0.10, 0.05, 0.15]

dict_indices = {row['id']: (row['ingreso_mensual'] / 1500) for _, row in df_usuarios.iterrows()}
categorias_elasticas = ['Vivienda', 'Educacion', 'Inversion', 'Ocio', 'Vestimenta', 'Electrodomesticos']

# Perfiles de comportamiento
usuarios_derrochadores = {1, 2}
usuarios_austeros = {3, 4}

# Segmentación mensual exacta de la ventana de 365 días (2025-08-22 al 2026-08-21)
meses_info = [
    {"mes_key": "2025-08", "start_d": 22, "end_d": 31, "dias": 10, "dias_mes": 31, "n_tx": 16},
    {"mes_key": "2025-09", "start_d": 1,  "end_d": 30, "dias": 30, "dias_mes": 30, "n_tx": 50},
    {"mes_key": "2025-10", "start_d": 1,  "end_d": 31, "dias": 31, "dias_mes": 31, "n_tx": 50},
    {"mes_key": "2025-11", "start_d": 1,  "end_d": 30, "dias": 30, "dias_mes": 30, "n_tx": 50},
    {"mes_key": "2025-12", "start_d": 1,  "end_d": 31, "dias": 31, "dias_mes": 31, "n_tx": 50},
    {"mes_key": "2026-01", "start_d": 1,  "end_d": 31, "dias": 31, "dias_mes": 31, "n_tx": 50},
    {"mes_key": "2026-02", "start_d": 1,  "end_d": 28, "dias": 28, "dias_mes": 28, "n_tx": 49},
    {"mes_key": "2026-03", "start_d": 1,  "end_d": 31, "dias": 31, "dias_mes": 31, "n_tx": 51},
    {"mes_key": "2026-04", "start_d": 1,  "end_d": 30, "dias": 30, "dias_mes": 30, "n_tx": 50},
    {"mes_key": "2026-05", "start_d": 1,  "end_d": 31, "dias": 31, "dias_mes": 31, "n_tx": 50},
    {"mes_key": "2026-06", "start_d": 1,  "end_d": 30, "dias": 30, "dias_mes": 30, "n_tx": 50},
    {"mes_key": "2026-07", "start_d": 1,  "end_d": 31, "dias": 31, "dias_mes": 31, "n_tx": 50},
    {"mes_key": "2026-08", "start_d": 1,  "end_d": 21, "dias": 21, "dias_mes": 31, "n_tx": 34},
]

transacciones = []
id_tx = 1

for _, usr in df_usuarios.iterrows():
    user_id = usr['id']
    sueldo_mensual = usr['ingreso_mensual']
    idx_adquisitivo = dict_indices[user_id]
    
    if user_id in usuarios_derrochadores:
        perfil_tipo = 'derrochador'
    elif user_id in usuarios_austeros:
        perfil_tipo = 'austero'
    else:
        perfil_tipo = 'estandar'
        
    for seg in meses_info:
        mes_key = seg["mes_key"]
        anio, mes_num = map(int, mes_key.split('-'))
        dias_activos = seg["dias"]
        dias_mes = seg["dias_mes"]
        n_tx = seg["n_tx"]
        
        if perfil_tipo == 'derrochador':
            ratio_base = np.random.uniform(0.92, 0.98)
        elif perfil_tipo == 'austero':
            ratio_base = np.random.uniform(0.45, 0.65)
        else:
            ratio_base = np.random.uniform(0.70, 0.85)
            
        sueldo_periodo = sueldo_mensual * (dias_activos / dias_mes)
        gasto_objetivo_periodo = round(sueldo_periodo * ratio_base, 2)
        
        cats_lote = list(np.random.choice(categorias, size=n_tx, p=prob_categorias))
        
        for i in range(n_tx):
            c = cats_lote[i]
            if perfil_tipo == 'derrochador' and c not in ['Vivienda', 'Servicios'] and np.random.rand() < 0.10:
                cats_lote[i] = np.random.choice(['Ocio', 'Vestimenta'])
            elif perfil_tipo == 'austero' and c not in ['Vivienda', 'Servicios'] and np.random.rand() < 0.08:
                cats_lote[i] = 'Inversion'
                
        dias_sorteados = sorted([random.randint(seg["start_d"], seg["end_d"]) for _ in range(n_tx)])
        
        montos_raw = []
        descripciones_lote = []
        
        for i in range(n_tx):
            cat = cats_lote[i]
            desc = np.random.choice(diccionario_conceptos[cat])
            
            if np.random.rand() < 0.20:
                prefijos = ["pago ", "compra ", "tarjeta ", "fac ", ""]
                desc = str(np.random.choice(prefijos)) + desc
            if np.random.rand() < 0.10:
                desc = desc.replace("a", "q", 1) if "a" in desc else desc.replace("e", "w", 1)
                
            min_m, max_m = rangos_montos[cat]
            
            if cat in categorias_elasticas:
                max_m = max_m * (1 + (idx_adquisitivo - 1) * 0.5)
                min_m = min_m * (1 + (idx_adquisitivo - 1) * 0.2)
            else:
                max_m = max_m * (1 + (idx_adquisitivo - 1) * 0.2)
                
            if cat in ['Vivienda', 'Servicios']:
                if perfil_tipo == 'derrochador':
                    max_m *= 1.8
                    min_m *= 1.8
                elif perfil_tipo == 'austero':
                    max_m *= 0.3
                    min_m *= 0.3
                    
            if cat in ['Servicios', 'Educacion', 'Vivienda']:
                moda_m = min_m + (max_m - min_m) * 0.5
            else:
                moda_m = min_m + (max_m - min_m) * 0.25
                
            monto_raw = float(np.random.triangular(min_m, moda_m, max_m))
            montos_raw.append(monto_raw)
            descripciones_lote.append(desc)
            
        suma_raw = sum(montos_raw)
        escala = gasto_objetivo_periodo / suma_raw
        montos_lote = [round(m * escala, 2) for m in montos_raw]
        
        diff = round(gasto_objetivo_periodo - sum(montos_lote), 2)
        montos_lote[0] = round(montos_lote[0] + diff, 2)
        
        for i in range(n_tx):
            fecha_str = f"{anio:04d}-{mes_num:02d}-{dias_sorteados[i]:02d}"
            transacciones.append({
                "id": id_tx,
                "descripcion": descripciones_lote[i],
                "monto": max(1.0, montos_lote[i]),
                "categoria": cats_lote[i],
                "fecha": fecha_str,
                "usuario_id": user_id
            })
            id_tx += 1

df_transacciones = pd.DataFrame(transacciones)
print(f"Se han generado {len(df_transacciones)} transacciones simuladas (2025-08-22 al 2026-08-21).")
display(df_transacciones.head())


## 3. Exportación de Datos
*   **CSV**: `usuarios.csv` y `transacciones.csv` para análisis.
*   **SQL**: `poblar_datos.sql` usando `INSERT IGNORE` y comentarios estándar `-- ` para asegurar compatibilidad e idempotencia en MySQL.


In [ ]:
# 1. Exportar a formato CSV
df_usuarios.to_csv(f"{output_dir}/usuarios.csv", index=False)
df_transacciones.to_csv(f"{output_dir}/transacciones.csv", index=False)
print("Archivos CSV generados exitosamente.")

# 2. Exportar a formato SQL con INSERT IGNORE y comentarios estandar
with open(f"{output_dir}/poblar_datos.sql", "w", encoding="utf-8") as f:
    f.write("-- Script de insercion masiva para FinanceAI (2025-08-22 al 2026-08-21)\n\n")
    
    # Usuarios con INSERT IGNORE
    for _, row in df_usuarios.iterrows():
        sql = f"INSERT IGNORE INTO usuarios (id, nombre, email, password, ingreso_mensual, fecha_creacion, activo) VALUES ({row['id']}, '{row['nombre']}', '{row['email']}', '{row['password']}', {row['ingreso_mensual']}, '{row['fecha_creacion']}', {row['activo']});\n"
        f.write(sql)
        
    f.write("\n-- Transacciones con INSERT IGNORE\n")
    for _, row in df_transacciones.iterrows():
        desc = row['descripcion'].replace("'", "''")
        sql = f"INSERT IGNORE INTO transacciones (id, descripcion, monto, categoria, fecha, usuario_id) VALUES ({row['id']}, '{desc}', {row['monto']}, '{row['categoria']}', '{row['fecha']}', {row['usuario_id']});\n"
        f.write(sql)

print(f"Archivo SQL generado en '{output_dir}/poblar_datos.sql'.")
